# Understanding the digitisation of the ODD

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
atl.ATLAS = "ColliderML"

sys.path.append("../")
from utils import load_root_file, load_hepmc_event
from edm4hep_utils import build_calo_df, build_particle_df, build_tracker_df

**Points of comparison**
- Hit by hit, what is the difference in positions between hits_df and measurements_df?

## Load ODD Data

In [12]:
acts_base_path = Path("/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_testing16")
hits_df = load_root_file(acts_base_path / "simhits.root")
smeared_measurements_df = load_root_file(acts_base_path / "measurements.root")
# calo_df = load_root_file(acts_base_path / "calohits.root")
particles_df = load_root_file(acts_base_path / "particles.root")

hits_df = hits_df[hits_df["event_id"] == 0]
# calo_df = calo_df[calo_df["event_id"] == 0]
smeared_measurements_df = smeared_measurements_df[smeared_measurements_df["event_nr"] == 0]
particles_df = particles_df[particles_df["event_id"] == 0]

In [13]:
acts_base_path = Path("/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_testing17")
geo_measurements_df = load_root_file(acts_base_path / "measurements.root")
geo_measurements_df = geo_measurements_df[geo_measurements_df["event_nr"] == 0]

In [14]:
smeared_measurements_df

,event_nr,volume_id,layer_id,surface_id,rec_loc0,rec_loc1,rec_time,var_loc0,var_loc1,var_time,...,true_y,true_z,true_incident_phi,true_incident_theta,residual_loc0,residual_loc1,residual_time,pull_loc0,pull_loc1,pull_time
entry,,,,,,,,,,,,,,,,,,,,,
0,0,16,4,1,-10.095659,-14.757201,-1.922955,0.000225,0.000225,625.0,...,10.087406,-1515.599976,-1.598269,-1.534504,-0.008253,0.007108,-22.920671,-0.550207,0.473849,-0.916827
1,0,16,4,1,9.577584,12.353203,1996.363892,0.000225,0.000225,625.0,...,-9.607749,-1515.599976,-1.570763,-1.511828,-0.030165,0.005889,1.995850,-2.010981,0.392596,0.079834
2,0,16,4,1,2.502311,18.356956,4059.385986,0.000225,0.000225,625.0,...,-2.501978,-1515.599976,-1.576217,-1.511414,0.000333,-0.007977,5.552002,0.022221,-0.531769,0.222080
3,0,16,4,1,-3.597521,-19.722897,1755.199585,0.000225,0.000225,625.0,...,3.599658,-1515.599976,-1.497631,-1.881402,0.002137,0.017015,-13.128052,0.142463,1.134364,-0.525122
4,0,16,4,1,-2.179284,14.025929,1741.227539,0.000225,0.000225,625.0,...,2.148883,-1515.599976,-1.483269,-1.411196,-0.030401,0.011687,28.255737,-2.026717,0.779152,1.130229
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209016,0,30,12,192,56.710541,NaN,NaN,0.005184,NaN,NaN,...,876.332214,3009.500000,-1.248641,-1.220544,-0.059521,NaN,NaN,-0.826677,NaN,NaN
209017,0,30,12,192,-35.503166,NaN,NaN,0.005184,NaN,NaN,...,980.081299,3009.500000,1.364493,1.264974,-0.009888,NaN,NaN,-0.137329,NaN,NaN
209018,0,30,12,192,-17.417934,NaN,NaN,0.005184,NaN,NaN,...,928.675659,3009.500000,1.571392,1.242740,-0.037735,NaN,NaN,-0.524097,NaN,NaN


In [15]:
geo_measurements_df

,event_nr,volume_id,layer_id,surface_id,rec_loc0,rec_loc1,rec_time,var_loc0,var_loc1,var_time,...,true_y,true_z,true_incident_phi,true_incident_theta,residual_loc0,residual_loc1,residual_time,pull_loc0,pull_loc1,pull_time
entry,,,,,,,,,,,,,,,,,,,,,
0,0,17,2,1,5.499168,-21.634134,NaN,0.000208,0.000208,NaN,...,5.243798,-492.884674,-1.055837,-2.609093,0.000344,0.000555,NaN,0.023853,0.038453,NaN
1,0,17,2,1,-0.441414,4.403106,NaN,0.000208,0.000208,NaN,...,-0.651501,-466.846619,1.102693,2.709590,-0.002237,-0.000278,NaN,-0.155010,-0.019293,NaN
2,0,17,2,1,-3.593196,21.350000,NaN,0.000208,0.000208,NaN,...,-3.785146,-449.902008,1.831124,3.082407,0.002325,0.002014,NaN,0.161101,0.139547,NaN
3,0,17,2,1,1.017173,21.567987,NaN,0.000208,0.000208,NaN,...,0.780948,-449.682159,1.655555,3.069601,0.013526,0.000137,NaN,0.937078,0.009514,NaN
4,0,17,2,1,6.774999,30.863565,NaN,0.000208,0.000208,NaN,...,6.514921,-440.390839,-1.229182,-2.631340,-0.004155,0.004398,NaN,-0.287877,0.304708,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106235,0,29,4,3360,-0.749999,21.750000,NaN,0.001875,0.187500,NaN,...,663.547180,-86.678986,0.769678,1.667356,-0.003236,-0.071014,NaN,-0.074729,-0.164001,NaN
106236,0,29,4,3360,15.225002,39.750000,NaN,0.001875,0.187500,NaN,...,677.073608,-69.050438,-2.246280,-1.569747,0.020986,0.300442,NaN,0.484637,0.693841,NaN
106237,0,29,4,3360,-16.124998,-11.250000,NaN,0.001875,0.187500,NaN,...,650.453308,-120.009727,2.687202,1.053509,0.012327,0.259726,NaN,0.284685,0.599811,NaN


In [16]:
hits_df

,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,
0,0,1152921779484754177,8022039789424584,61.235691,10.087406,-1515.599976,20.997717,0.027367,0.020712,-0.753723,...,0.0,0.0,0.0,-0.000035,-1,16,0,4,0,1
1,0,1152921779484754177,5468975769351603,88.347313,-9.607749,-1515.599976,1994.368042,0.222337,-0.000124,-3.766065,...,0.0,0.0,0.0,-0.000023,-1,16,0,4,0,1
2,0,1152921779484754177,8984110148358541,94.364929,-2.501978,-1515.599976,4053.833984,0.224067,0.020431,-3.768838,...,0.0,0.0,0.0,-0.000042,-1,16,0,4,0,1
3,0,1152921779484754177,6232036235374477,56.260086,3.599658,-1515.599976,1768.327637,-0.000281,-0.000064,-0.000874,...,0.0,0.0,0.0,-0.000039,-1,16,0,4,0,1
4,0,1152921779484754177,6232036235302319,90.014244,2.148883,-1515.599976,1712.971802,0.015975,-0.008708,-0.099239,...,0.0,0.0,0.0,-0.000045,-1,16,0,4,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209021,0,2161728645771608066,6911535160008404,424.833771,876.332214,3009.500000,4069.588135,0.000151,0.000075,-0.000341,...,0.0,0.0,0.0,-0.000155,-1,30,0,12,0,192
209022,0,2161728645771608066,7298558790029736,366.332397,980.081299,3009.500000,4255.901367,0.447207,0.314389,1.443166,...,0.0,0.0,0.0,-0.000070,-1,30,0,12,0,192
209023,0,2161728645771608066,6323294459035673,365.682526,928.675659,3009.500000,5061.892090,0.039010,0.100295,0.316183,...,0.0,0.0,0.0,-0.000074,-1,30,0,12,0,192


In [19]:
smeared_measurements_df[["true_x", "true_y", "true_z"]]

,true_x,true_y,true_z
entry,,,
0,61.235691,10.087406,-1515.599976
1,88.347313,-9.607749,-1515.599976
2,94.364929,-2.501978,-1515.599976
3,56.260086,3.599658,-1515.599976
4,90.014244,2.148883,-1515.599976
...,...,...,...
209016,424.833771,876.332214,3009.500000
209017,366.332397,980.081299,3009.500000
209018,365.682526,928.675659,3009.500000


In [ ]:
geo_measurements_df[["true_x", "true_y", "true_z"]]

,true_x,true_y,true_z
entry,,,
0,31.554668,5.243798,-492.884674
1,32.265518,-0.651501,-466.846619
2,32.643372,-3.785146,-449.902008
3,32.092796,0.780948,-449.682159
4,31.401398,6.514921,-440.390839
...,...,...,...
106235,778.810181,663.547180,-86.678986
106236,770.356567,677.073608,-69.050438
106237,786.898621,650.453308,-120.009727


In [21]:
# Merge geo measurements and hits on true_x, true_y, true_z
shared_measurements_df = geo_measurements_df.merge(hits_df, left_on=["true_x", "true_y", "true_z"], right_on=["tx", "ty", "tz"], how="inner")



In [23]:
shared_measurements_df[["true_x", "true_y", "true_z", "tx", "ty", "tz"]]

,true_x,true_y,true_z,tx,ty,tz
0,31.554668,5.243798,-492.884674,31.554668,5.243798,-492.884674
1,32.265518,-0.651501,-466.846619,32.265518,-0.651501,-466.846619
2,32.643372,-3.785146,-449.902008,32.643372,-3.785146,-449.902008
3,32.092796,0.780948,-449.682159,32.092796,0.780948,-449.682159
4,31.401398,6.514921,-440.390839,31.401398,6.514921,-440.390839
...,...,...,...,...,...,...
106237,778.810181,663.547180,-86.678986,778.810181,663.547180,-86.678986
106238,770.356567,677.073608,-69.050438,770.356567,677.073608,-69.050438
106239,786.898621,650.453308,-120.009727,786.898621,650.453308,-120.009727
106240,774.138123,671.051575,-113.228661,774.138123,671.051575,-113.228661
